# Auditing a model on `diabetes130`

A full isitfair audit of a logistic regression fitted to the public
`diabetes130` dataset: discrimination, calibration, decision curves,
Hosmer-Lemeshow, estimability, a recalibration ladder, mitigation and an
HTML report.

Before running, build the prediction table once from the repository root:

    python -I extra_examples/datasets.py diabetes130

`B = 50` bootstrap replicates keep the run short; use 2,000 for results you
intend to report.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

for _c in (Path.cwd(), *Path.cwd().parents):
    if (_c / "extra_examples" / "datasets.py").exists():
        HERE = _c / "extra_examples"
        break
sys.path.insert(0, str(HERE))
import datasets  # noqa: E402

import isitfair  # noqa: E402
from isitfair import FairnessAudit, GroupRecalibration, WassersteinPostprocessing  # noqa: E402

NAME = "diabetes130"
B = 50
SEED = 42
META = datasets.meta(NAME)
THRESHOLD = META["threshold"]
ATTRS = datasets.ATTRS[NAME]
print(f"isitfair {isitfair.__version__}")
print(f"{NAME}: outcome {META['outcome']}; threshold {THRESHOLD}; B = {B}")

isitfair 0.4.0
diabetes130: outcome readmitted == '<30' (readmission within 30 days); threshold 0.15; B = 50


## 1. Load the predictions

One row per test-set patient or person: the outcome, the model's predicted
probability, and the attributes to audit.

In [2]:
df = datasets.load(NAME)
print(f"n = {len(df)}, prevalence = {df['y_true'].mean():.3f}")
for a in ATTRS:
    print(f"{a}: {df[a].nunique()} levels, {int(df[a].isna().sum())} missing")

n = 30530, prevalence = 0.112
race: 5 levels, 676 missing
gender: 3 levels, 0 missing
age: 10 levels, 0 missing


isitfair refuses missing subgroup labels rather than silently dropping those
rows. If any attribute has missing labels, this cell shows the error and then
recodes them to an explicit `"Unknown"` level.

In [3]:
missing_attrs = [a for a in ATTRS if df[a].isna().any()]
if missing_attrs:
    try:
        FairnessAudit(y_true=df["y_true"], y_proba=df["y_proba"],
                      sensitive_features={a: df[a] for a in ATTRS},
                      threshold=THRESHOLD, n_bootstrap=B, random_state=SEED)
    except ValueError as exc:
        print("ValueError:", exc)
    for a in missing_attrs:
        df[a] = df[a].fillna("Unknown")
print("recoded to 'Unknown':", missing_attrs or "none")

ValueError: `sensitive_features['race']` has 676 missing value(s). Subgroup labels cannot be missing: drop those rows from every input, or recode them as an explicit category, e.g. `df[col].fillna('unknown')`.
recoded to 'Unknown': ['race']


## 2. Split off a calibration half for mitigation

The test half is split 50/50, stratified on the outcome. Mitigation
calibrators are fitted on one half and every audit runs on the other, so no
row is scored by a calibrator fitted on it.

In [4]:
i_cal, i_eval = train_test_split(np.arange(len(df)), test_size=0.5, random_state=SEED,
                                 stratify=df["y_true"])
cal_df, eval_df = df.iloc[i_cal].reset_index(drop=True), df.iloc[i_eval].reset_index(drop=True)
# mitigate() rejects audited levels that the calibration half never saw; drop
# such rows (for example diabetes130's single "Unknown/Invalid" gender).
unseen = np.zeros(len(eval_df), bool)
for a in ATTRS:
    unseen |= ~eval_df[a].isin(set(cal_df[a]))
if unseen.any():
    print(f"dropping {int(unseen.sum())} audited row(s) with levels absent from the calibration half")
    eval_df = eval_df[~unseen].reset_index(drop=True)
print(f"calibration n = {len(cal_df)}, audit n = {len(eval_df)}")

dropping 1 audited row(s) with levels absent from the calibration half
calibration n = 15265, audit n = 15264


## 3. Construct the audit

Each attribute is audited on its own, and the first two are also crossed
(passing `None` for a tuple key asks isitfair to build the intersection).

In [5]:
y = eval_df["y_true"].to_numpy(dtype=float)
p = eval_df["y_proba"].to_numpy(dtype=float)
marginals = {a: eval_df[a].to_numpy(dtype=object) for a in ATTRS}
INTERSECTION = tuple(ATTRS[:2]) if len(ATTRS) >= 2 else None
sf = dict(marginals)
if INTERSECTION:
    sf[INTERSECTION] = None

audit = FairnessAudit(y_true=y, y_proba=p, sensitive_features=sf, threshold=THRESHOLD,
                      n_bootstrap=B, random_state=SEED)
audit

FairnessAudit(n=15264, events=1703, threshold=0.15, n_bootstrap=50, random_state=42, attributes: race (6), gender (2), age (10), race × gender (12))

## 4. Discrimination

In [6]:
disc = audit.discrimination()
disc[disc.subgroup != "GAP"][["attribute", "subgroup", "n", "n_events", "auroc",
                              "auroc_ci_low", "auroc_ci_high"]]

,attribute,subgroup,n,n_events,auroc,auroc_ci_low,auroc_ci_high
0,race,Overall,15264.0,1703.0,0.633490,0.619615,0.647385
1,race,AfricanAmerican,2888.0,316.0,0.638305,0.607536,0.669213
2,race,Asian,82.0,4.0,0.516026,NaN,NaN
3,race,Caucasian,11479.0,1307.0,0.629759,0.615279,0.642062
4,race,Hispanic,277.0,32.0,0.721429,0.649416,0.799999
5,race,Other,213.0,11.0,0.634113,NaN,NaN
6,race,Unknown,325.0,33.0,0.641137,0.552657,0.704495
11,gender,Overall,15264.0,1703.0,0.633490,0.619615,0.647385
12,gender,Female,8168.0,922.0,0.643916,0.620941,0.664985
13,gender,Male,7096.0,781.0,0.621109,0.602602,0.637476


## 5. Calibration

In [7]:
cal = audit.calibration()
cal[cal.subgroup != "GAP"][["attribute", "subgroup", "n", "calibration_intercept",
                            "calibration_slope", "brier", "ece", "ici"]]

,attribute,subgroup,n,calibration_intercept,calibration_slope,brier,ece,ici
0,race,Overall,15264.0,-0.001528,0.789849,0.096337,0.011320,0.009375
1,race,AfricanAmerican,2888.0,-0.044023,0.813347,0.094199,0.017285,0.010862
2,race,Asian,82.0,-0.814003,0.099986,0.052287,0.078907,0.054083
3,race,Caucasian,11479.0,0.018193,0.766074,0.098300,0.012270,0.011342
4,race,Hispanic,277.0,0.100511,1.411816,0.091441,0.045640,0.029669
5,race,Other,213.0,-0.722051,0.836755,0.051416,0.059477,0.048019
6,race,Unknown,325.0,0.059725,0.812896,0.090744,0.053805,0.031703
9,gender,Overall,15264.0,-0.001528,0.789849,0.096337,0.011320,0.009375
10,gender,Female,8168.0,-0.001346,0.805150,0.097012,0.013971,0.008284
11,gender,Male,7096.0,-0.001740,0.768671,0.095561,0.016393,0.012178


In [8]:
curves = audit.calibration_curves()
roc = audit.roc_curves()
print("calibration curves:", curves.shape, "| ROC curves:", roc.shape)

calibration curves: (3400, 7) | ROC curves: (3400, 7)


## 6. Decision curve: gaps in net benefit between subgroups

In [9]:
dc = audit.decision_curve()
dc[dc.subgroup == "GAP"][["attribute", "gap_value", "gap_ci_low", "gap_ci_high"]]

,attribute,gap_value,gap_ci_low,gap_ci_high
140,race,0.078732,NaN,NaN
201,gender,0.008284,0.004913,0.016004
422,age,0.156353,NaN,NaN
683,race × gender,0.119896,NaN,NaN


## 7. Estimability

Cells with too few events for a metric are flagged rather than reported.

In [10]:
est = audit.estimability()
print(f"{int(est.not_estimable_final.sum())} of {len(est)} cell-metrics not reportable")

149 of 402 cell-metrics not reportable


## 8. Hosmer-Lemeshow

In [11]:
hl = audit.hosmer_lemeshow()
hl[hl.subgroup != "GAP"][["attribute", "subgroup", "n", "hl_statistic", "hl_df",
                          "hl_p_value", "n_bins_used"]]

,attribute,subgroup,n,hl_statistic,hl_df,hl_p_value,n_bins_used
0,race,Overall,15264.0,33.227545,10.0,0.000249,10.0
1,race,AfricanAmerican,2888.0,13.373135,10.0,0.203551,10.0
2,race,Asian,82.0,7.320330,10.0,0.694897,10.0
3,race,Caucasian,11479.0,28.020401,10.0,0.001792,10.0
4,race,Hispanic,277.0,10.924993,10.0,0.363392,10.0
5,race,Other,213.0,10.957537,10.0,0.360836,10.0
6,race,Unknown,325.0,15.140458,10.0,0.127024,10.0
8,gender,Overall,15264.0,33.227545,10.0,0.000249,10.0
9,gender,Female,8168.0,21.452352,10.0,0.018151,10.0
10,gender,Male,7096.0,26.109123,10.0,0.003596,10.0


## 9. Intersection cells

In [12]:
audit.cell_diagnostics() if INTERSECTION else "single attribute; no intersection"

,attribute,subgroup,n,n_events,shrunk,marginal_rate,cell_rate,shrinkage_factor
0,race × gender,AfricanAmerican / Female,1750,194,False,0.109418,0.110857,0.244930
1,race × gender,AfricanAmerican / Male,1138,122,False,0.109418,0.107206,0.332812
2,race × gender,Asian / Female,41,3,True,0.048780,0.073171,0.932640
3,race × gender,Asian / Male,41,1,True,0.048780,0.024390,0.932640
4,race × gender,Caucasian / Female,5952,688,False,0.113860,0.115591,0.087070
5,race × gender,Caucasian / Male,5527,619,False,0.113860,0.111996,0.093142
6,race × gender,Hispanic / Female,151,18,True,0.115523,0.119205,0.789889
7,race × gender,Hispanic / Male,126,14,True,0.115523,0.111111,0.818357
8,race × gender,Other / Female,105,3,True,0.051643,0.028571,0.843905
9,race × gender,Other / Male,108,8,True,0.051643,0.074074,0.840158


## 10. Recalibration ladder on the attribute with the largest intercept gap

A level with no events at all cannot be recalibrated, and isitfair says so.
In that case the ladder moves on to the next attribute.

In [13]:
gaps = cal[(cal.subgroup == "GAP") & (cal.gap_metric == "max_intercept_gap")
           & cal.attribute.isin(ATTRS)]
RANKED = [str(a) for a in gaps.sort_values("gap_value", ascending=False).attribute]
ladder = None
for by in RANKED:
    try:
        ladder = audit.recalibration_ladder(by=by)
        print("ladder by", by)
        break
    except ValueError as exc:
        print(f"{by}: {exc}")
ladder

ladder by race


,rung,attribute,subgroup,n,n_events,calibration_intercept,intercept_ci_low,intercept_ci_high,calibration_slope,slope_ci_low,...,brier_ci_low,brier_ci_high,ece,ici,net_benefit,gap_metric,gap_value,event_floor,below_floor,not_estimable_final
0,none,race,Overall,15264.0,1703.0,-0.001528,-0.050273,0.040091,0.789849,0.710610,...,0.093092,0.099488,0.011320,0.009375,0.010120,NaN,NaN,25.0,False,False
1,none,race,AfricanAmerican,2888.0,316.0,-0.044023,-0.170808,0.050724,0.813347,0.648900,...,0.086377,0.099646,0.017285,0.010862,0.013463,NaN,NaN,25.0,False,False
2,none,race,Asian,82.0,4.0,-0.814003,NaN,NaN,0.099986,NaN,...,NaN,NaN,0.078907,0.054083,-0.015782,NaN,NaN,25.0,True,True
3,none,race,Caucasian,11479.0,1307.0,0.018193,-0.035497,0.083569,0.766074,0.718428,...,0.094894,0.102663,0.012270,0.011342,0.009808,NaN,NaN,25.0,False,False
4,none,race,Hispanic,277.0,32.0,0.100511,-0.434019,0.417579,1.411816,NaN,...,0.062447,0.116691,0.045640,0.029669,0.031217,NaN,NaN,25.0,False,True
5,none,race,Other,213.0,11.0,-0.722051,NaN,NaN,0.836755,NaN,...,NaN,NaN,0.059477,0.048019,-0.008837,NaN,NaN,25.0,True,True
6,none,race,Unknown,325.0,33.0,0.059725,-0.329436,0.349667,0.812896,NaN,...,0.065848,0.113687,0.053805,0.031703,-0.007602,NaN,NaN,25.0,False,True
7,none,race,GAP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,max_intercept_gap,0.914514,NaN,NaN,NaN
8,common,race,Overall,15264.0,1703.0,0.001459,-0.044553,0.040080,0.890376,0.749775,...,0.092997,0.099294,0.005438,0.002496,0.010024,NaN,NaN,25.0,False,False
9,common,race,AfricanAmerican,2888.0,316.0,-0.030855,-0.153368,0.056207,0.987219,0.788766,...,0.086079,0.099616,0.013625,0.006460,0.012425,NaN,NaN,25.0,False,False


## 11. Mitigation

Group-wise isotonic recalibration, then Wasserstein post-processing, both
fitted on the calibration half.

In [14]:
calib_data = (cal_df["y_true"].to_numpy(float), cal_df["y_proba"].to_numpy(float),
              {a: cal_df[a].to_numpy(object) for a in ATTRS})
mitigated_gr = audit.mitigate(method=GroupRecalibration(method="isotonic"), calibration_data=calib_data)
a0 = ATTRS[0]
keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before, after = audit.calibration(), mitigated_gr[a0].calibration()
print("Before,", a0)
print(before[before.attribute == a0][keep].to_string(index=False))
print("\nAfter group isotonic recalibration,", a0)
print(after[after.attribute == a0][keep].to_string(index=False))

Before, race
       subgroup  calibration_intercept  calibration_slope    brier      ece
        Overall              -0.001528           0.789849 0.096337 0.011320
AfricanAmerican              -0.044023           0.813347 0.094199 0.017285
          Asian              -0.814003           0.099986 0.052287 0.078907
      Caucasian               0.018193           0.766074 0.098300 0.012270
       Hispanic               0.100511           1.411816 0.091441 0.045640
          Other              -0.722051           0.836755 0.051416 0.059477
        Unknown               0.059725           0.812896 0.090744 0.053805
            GAP                    NaN                NaN      NaN      NaN
            GAP                    NaN                NaN      NaN      NaN

After group isotonic recalibration, race
       subgroup  calibration_intercept  calibration_slope    brier      ece
        Overall              -0.001047           0.587090 0.096440 0.011267
AfricanAmerican              -0.0

In [15]:
mitigated_w = audit.mitigate(method=WassersteinPostprocessing(random_state=SEED), calibration_data=calib_data)
d_after = mitigated_w[a0].discrimination()
d_after[(d_after.attribute == a0) & (d_after.subgroup == "GAP")][
    ["gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]

,gap_metric,gap_value,gap_ci_low,gap_ci_high
7,equal_opportunity_difference,0.218750,NaN,NaN
8,predictive_equality,0.075092,NaN,NaN
9,equalized_odds_gap,0.218750,NaN,NaN
10,statistical_parity,0.050498,NaN,NaN


## 12. HTML report

In [16]:
REPORT = HERE / "reports" / f"{NAME}.html"
REPORT.parent.mkdir(exist_ok=True)
audit.report(out=REPORT, title=f"{NAME}: isitfair audit",
             metadata={"Dataset": NAME, "Model": "LogisticRegression",
                       "Threshold": str(THRESHOLD), "Bootstrap": f"B={B}, random_state={SEED}"},
             mitigation_audits=mitigated_gr)
print("wrote", REPORT.relative_to(HERE.parent))

wrote extra_examples\reports\diabetes130.html
